# Session 1 – Spustenie Chat Bootstrap (Foundry Local)

Tento notebook spúšťa Foundry Local, sťahuje preferovaný alias modelu a vykonáva štandardné aj streamované dokončenie chatu.


# Scenár
Táto relácia predstavuje absolútne minimum na to, aby malý lokálny jazykový model reagoval prostredníctvom Foundry Local. Budete:
- Nainštalovať SDK / klientské závislosti.
- Inicializovať správcu Foundry Local pre zvolený alias (predvolené: `phi-4-mini`).
- Použiť obranný monkey‑patch na tolerovanie voliteľných polí v metadátach modelu.
- Odoslať štandardnú požiadavku na dokončenie chatu.
- Streamovať odpoveď token po tokenu.

Cieľom je overiť váš lokálny runtime a sieťovú cestu pred prechodom na RAG, smerovanie alebo agentov.


### Vysvetlenie: Inštalácia závislostí
Inštaluje Python balíky potrebné pre tento jednoduchý chatovací tok:
- `foundry-local-sdk`: Správa lokálnych modelov a životného cyklu služieb.
- `openai`: Známa abstrakcia klienta pre dokončenie chatov.
- `rich`: Pekné formátovanie pre prehľadnejší výstup v notebooku.

Opätovné spustenie je bezpečné (idempotentné). Preskočte, ak už vaše prostredie tieto balíky obsahuje.


In [1]:
# Install required libraries (idempotent)
%pip install -q foundry-local-sdk openai rich


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Vysvetlenie: Základné importy
Zahŕňa moduly používané v celom notebooku:
- `FoundryLocalManager` na interakciu s lokálnym runtime modelu.
- Klient `OpenAI`, aby sme mohli používať známe rozhranie API pre chatové dokončovanie.
- `rich.print` pre štýlový výstup.

Tu sa nevykonávajú žiadne sieťové volania—ide len o prípravu názvového priestoru.


In [2]:
import os
from foundry_local import FoundryLocalManager
from foundry_local.models import FoundryModelInfo
from openai import OpenAI
from rich import print

### Vysvetlenie: Inicializácia manažéra a úprava metadát
Inicializuje `FoundryLocalManager` pre zvolený alias a aplikuje defenzívnu úpravu na elegantné spracovanie odpovedí služby, kde `promptTemplate` môže byť `null`.

Hlavné výsledky:
- Potvrdzuje stav služby a koncový bod.
- Zobrazuje uložené modely (overuje lokálny úložisko).
- Určuje konkrétne ID modelu pre alias (používané v neskorších volaniach chatu).

Ak narazíte na problémy s validáciou v surových metadátach služby, tento vzor ukazuje, ako ich vyčistiť bez potreby úpravy SDK.


In [3]:
# Monkeypatch to tolerate service responses where promptTemplate is null
_original_from_list_response = FoundryModelInfo.from_list_response

def _safe_from_list_response(response):  # type: ignore
    try:
        if isinstance(response, dict) and response.get("promptTemplate") is None:
            # Normalize to empty dict so pydantic validation passes
            response["promptTemplate"] = {}
    except Exception as e:  # pragma: no cover
        print(f"[yellow]Warning: safe wrapper encountered issue normalizing promptTemplate: {e}[/yellow]")
    return _original_from_list_response(response)

# Apply patch only once
if getattr(FoundryModelInfo.from_list_response, "__name__", "") != "_safe_from_list_response":
    FoundryModelInfo.from_list_response = staticmethod(_safe_from_list_response)  # type: ignore

ALIAS = os.getenv('FOUNDRY_LOCAL_ALIAS', 'phi-4-mini')
manager = FoundryLocalManager(ALIAS)
print(f'[bold green]Service running:[/bold green] {manager.is_service_running()}')
print(f'Endpoint: {manager.endpoint}')
print('Cached models:', manager.list_cached_models())
model_id = manager.get_model_info(ALIAS).id
print(f'Using model id: {model_id}')

Service running: True

Endpoint: http://127.0.0.1:50262/v1

Cached models:
[
    FoundryModelInfo(
        alias='phi-4-mini',
        id='Phi-4-mini-instruct-generic-gpu:4',
        version='4',
        execution_provider='WebGpuExecutionProvider',
        device_type=<DeviceType.GPU: 'GPU'>,
        uri='azureml://registries/azureml/models/Phi-4-mini-instruct-generic-gpu/versions/4',
        file_size_mb=3809,
        prompt_template={
            'system': '<|system|>{Content}<|end|>',
            'user': '<|user|>{Content}<|end|>',
            'assistant': '<|assistant|>{Content}<|end|>',
            'prompt': '<|user|>{Content}<|end|><|assistant|>'
        },
        provider='AzureFoundry',
        publisher='Microsoft',
        license='MIT',
        task='chat-completion',
        ep_override=None
    ),
    FoundryModelInfo(
        alias='qwen2.5-0.5b',
        id='qwen2.5-0.5b-instruct-generic-gpu:3',
        version='3',
        execution_provider='WebGpuExecutionProvider',
        device_type=<DeviceType.GPU: 'GPU'>,
        uri='azureml://registries/azureml/models/qwen2.5-0.5b-instruct-generic-gpu/versions/3',
        file_size_mb=700,
        prompt_template={
            'system': '<|im_start|>system\n{Content}<|im_end|>',
            'user': '<|im_start|>user\n{Content}<|im_end|>',
            'assistant': '<|im_start|>assistant\n{Content}<|im_end|>',
            'prompt': '<|im_start|>user\n{Content}<|im_end|>\n<|im_start|>assistant'
        },
        provider='AzureFoundry',
        publisher='Microsoft',
        license='apache-2.0',
        task='chat-completion',
        ep_override=None
    ),
    FoundryModelInfo(
        alias='phi-3.5-mini',
        id='Phi-3.5-mini-instruct-generic-gpu:1',
        version='1',
        execution_provider='WebGpuExecutionProvider',
        device_type=<DeviceType.GPU: 'GPU'>,
        uri='azureml://registries/azureml/models/Phi-3.5-mini-instruct-generic-gpu/versions/1',
        file_size_mb=2211,
        prompt_template={
            'prompt': '<|user|>\n{Content}<|end|>\n<|assistant|>',
            'assistant': '<|assistant|>\n{Content}<|end|>'
        },
        provider='AzureFoundry',
        publisher='Microsoft',
        license='MIT',
        task='chat-completion',
        ep_override=None
    )
]

Using model id: Phi-4-mini-instruct-generic-gpu:4

### Vysvetlenie: Základné dokončenie chatu
Vytvorí klienta kompatibilného s `OpenAI`, ktorý smeruje na lokálny Foundry endpoint a vykoná jedno dokončenie chatu bez streamovania. Zameranie tu:
- Zabezpečiť, aby model odpovedal bez chyby.
- Overiť latenciu / formát výstupu.
- Udržať `max_tokens` skromné, aby sa šetrili zdroje.

Ak toto zlyhá, znova skontrolujte, či je služba Foundry Local spustená a alias sa správne rozpoznáva.


In [4]:
client = OpenAI(base_url=manager.endpoint, api_key=manager.api_key or 'not-needed')
prompt = 'List two benefits of local inference for privacy.'
resp = client.chat.completions.create(
    model=model_id,
    messages=[{'role':'user','content':prompt}],
    max_tokens=120,
    temperature=0.5
)
print(resp.choices[0].message.content)

Local inference for privacy refers to the practice of performing data analysis on a local device without sending 
sensitive information to a central server. Two benefits of this approach are:


1. **Enhanced Privacy**: Local inference keeps personal data on the user's device, reducing the risk of data 
breaches and unauthorized access. Since the data is not transmitted over the network, it is less susceptible to 
interception by malicious actors.


2. **Data Sovereignty**: Users retain control over their data, as it does not leave their device. This means that 
individuals or organizations can comply with local data protection regulations, such as the General

### Vysvetlenie: Streamovanie dokončenia chatu
Ukazuje streamovanie tokenov pre zlepšenie vnímaného oneskorenia a interaktívneho používateľského zážitku. Slučka tlačí postupné zmeny, ako prichádzajú:
- Užitočné pre chatové rozhrania, kde záleží na skorom čiastočnom výstupe.
- Umožňuje merať priepustnosť tokenov oproti latencii úplného dokončenia.

Tento vzor môžete prispôsobiť na zhromažďovanie tokenov, aktualizáciu widgetu pokroku alebo prerušenie generovania v polovici.


In [5]:
# Streaming example
stream = client.chat.completions.create(
    model=model_id,
    messages=[{'role':'user','content':'Give a one-sentence definition of edge AI.'}],
    stream=True,
    max_tokens=60,
    temperature=0.4
)
for chunk in stream:
    delta = chunk.choices[0].delta
    if delta and delta.content:
        print(delta.content, end='', flush=True)
print()

Edge

AI

refers

to

artificial

...

latency

and

bandwidth

usage

.

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zrieknutie sa zodpovednosti**:  
Tento dokument bol preložený pomocou služby AI prekladu [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, prosím, berte na vedomie, že automatizované preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho rodnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nenesieme zodpovednosť za akékoľvek nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
